## Downloading flood conditioning variables

## Elevation, slope

In [66]:
import ee
import numpy as np
import time

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

def tile_name_from_latlon(lat, lon):

    """
    Example usage:
    Nairobi coordinates: 0.0236° S, 37.9062° E
    print(f"Nairobi tile: {tile_name_from_latlon(lat = 1.29, lon = 36.82)}")
    """

    # Data is in 3 x 3 degree tiles so
     
    lat_tile = (lat // 3) * 3
    lon_tile = (lon // 3) * 3
    lat_tile, lon_tile = int(lat_tile), int(lon_tile)

    lat_prefix = 'N' if lat_tile >= 0 else 'S'
    lon_prefix = 'E' if lon_tile >= 0 else 'W' 

    return f"{lat_prefix}{abs(lat_tile):02d}{lon_prefix}{abs(lon_tile):03d}"

def get_country_bonds(country_name):
    """
    Returns the bounding box of a country as [min_lon, min_lat, max_lon, max_lat]
    """
    countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    country = countries.filter(ee.Filter.eq('country_na', country_name))
    bounds = np.array(country.geometry().bounds().getInfo()['coordinates'][0])

    # Get min and max latitudes and longitudes
    min_lon, max_lon = bounds[:, 0].min(), bounds[:, 0].max()
    min_lat, max_lat = bounds[:, 1].min(), bounds[:, 1].max()
    return [min_lon, min_lat, max_lon, max_lat]

def generate_tiles_from_bounds(bounds, tile_size=3):
    """
    Generates a list of tile names that cover the given bounding box.
    """
    min_lon, min_lat, max_lon, max_lat = bounds

    # Align the bounds to the tile grid
    start_lon = (min_lon // tile_size) * tile_size
    end_lon = (max_lon // tile_size) * tile_size
    start_lat = (min_lat // tile_size) * tile_size
    end_lat = (max_lat // tile_size) * tile_size

    tiles = []
    for lat in range(int(start_lat), int(end_lat) + tile_size, tile_size):
        for lon in range(int(start_lon), int(end_lon) + tile_size, tile_size):
            tile = (lon, lat, lon + tile_size, lat + tile_size)
            tiles.append(tile)
            
    return tiles

def download_data_for_tile(tile, image, dataset_name, export_params):
    """
    Exports data for a given tile and dataset to Google Drive.
    """
    min_lon, min_lat, max_lon, max_lat = tile
    tile_name = tile_name_from_latlon(min_lat, min_lon)
    

    tile_geom = ee.Geometry.Rectangle(tile)
    export_params['region'] = tile_geom

    # Check if a task with the same description already exists
    tasks = ee.batch.Task.list()

    task_exists = any(
    ((t.config.get('description') == f"{tile_name}_{dataset_name}") and
    (t.state in ['READY', 'RUNNING'])) or
    ((t.config.get('description') == f"{tile_name}_{dataset_name}") and
    (t.state == 'COMPLETED'))
    for t in tasks
    )

    if task_exists:
        print(f"Task for {tile_name} and dataset {dataset_name} already exists. Skipping export.")
        return True

    print(f"Exporting tile: {tile_name} for dataset: {dataset_name}")
    task = ee.batch.Export.image.toDrive(
        image = image,
        description = f"{tile_name}_{dataset_name}",
        **export_params
    )
    task.start()
    return True

def check_task_status(n = 50):
    tasks = ee.batch.Task.list()
    print(f"{'TASK DESCRIPTION':<30} | {'STATE':<10} | {'ID'}")
    print("-" * 60)
    for task in tasks[:n]:  # Check the first n tasks
        status = task.status()
        description = status.get('description', 'No Description')
        state = status.get('state', 'UNKNOWN')
        task_id = status.get('id')
        print(f"{description:<30} | {state:<10} | {task_id}")
    

In [67]:
# Get the bounding box for Kenya
kenya_bounds = get_country_bonds("Kenya")

# Generate 3x3 degree tiles that cover Kenya
kenya_tiles = generate_tiles_from_bounds(kenya_bounds, tile_size=3)

# Define global datasets
# DEM: SRTM V3 (30m)
srtm = ee.Image("USGS/SRTMGL1_003")
dem_global = srtm.select('elevation')

# Slope: Derived from SRTM
slope_global = ee.Terrain.slope(dem_global)

# Define export parameters
export_params = {
    'scale': 30, # Force 30m resolution for ALL layers
    'crs': 'EPSG:4326',
    'maxPixels': 1e13,
    'fileFormat': 'GeoTIFF',
    'folder': 'Kenya_Flood_Data_3x3' # Creates this folder in Drive
}

# Download DEM and Slope for each tile
for tile in kenya_tiles:
    download_data_for_tile(tile, dem_global, "DEM", export_params)
    download_data_for_tile(tile, slope_global, "Slope", export_params)
    time.sleep(10)

# Check task status
check_task_status()


Task for S06E033 and dataset DEM already exists. Skipping export.
Exporting tile: S06E033 for dataset: Slope
Task for S06E036 and dataset DEM already exists. Skipping export.
Exporting tile: S06E036 for dataset: Slope
Task for S06E039 and dataset DEM already exists. Skipping export.
Exporting tile: S06E039 for dataset: Slope
Task for S03E033 and dataset DEM already exists. Skipping export.
Exporting tile: S03E033 for dataset: Slope
Task for S03E036 and dataset DEM already exists. Skipping export.
Exporting tile: S03E036 for dataset: Slope
Task for S03E039 and dataset DEM already exists. Skipping export.
Exporting tile: S03E039 for dataset: Slope
Task for N00E033 and dataset DEM already exists. Skipping export.
Exporting tile: N00E033 for dataset: Slope
Task for N00E036 and dataset DEM already exists. Skipping export.
Exporting tile: N00E036 for dataset: Slope
Task for N00E039 and dataset DEM already exists. Skipping export.
Exporting tile: N00E039 for dataset: Slope
Task for N03E033 an

In [69]:
check_task_status()

TASK DESCRIPTION               | STATE      | ID
------------------------------------------------------------
N03E039_Slope                  | COMPLETED  | LVLSDOJSAWN7PP3YJEM4H25M
N03E036_Slope                  | COMPLETED  | ZX4UWLQQVHCCR72CNRVZACWZ
N03E033_Slope                  | COMPLETED  | BMBAOQEKKYVXNSRAYR4K3YR2
N00E039_Slope                  | COMPLETED  | VM6YX4MNA7CRVM7UR3NJMFWT
N00E036_Slope                  | COMPLETED  | LZWBBVEBVVYANXSUGMDUWOVG
N00E033_Slope                  | COMPLETED  | TG4CEIF37TNCIT6DJHVFF62C
S03E039_Slope                  | COMPLETED  | 7H7B5ODYGZCZ3SO2434XAG67
S03E036_Slope                  | COMPLETED  | F6MAOBIFPNY2FNWR5T76XWSJ
S03E033_Slope                  | COMPLETED  | HJ2AW5KKV76MJV5I3PW3YCLR
S06E039_Slope                  | COMPLETED  | O7TTADZU2TE7CEDHI5LRBFC5
S06E036_Slope                  | COMPLETED  | GJEUSOAFAZUVVTLOBFGDF7PU
S06E033_Slope                  | COMPLETED  | SQ2X5YDG5N4IX6WYNU6V2QAW
N03E039_DEM                    | COMPL

## Precipitation — CHIRPS Daily (aggregated to monthly)
- **Source:** `UCSB-CHG/CHIRPS/DAILY` on GEE
- **Native resolution:** 0.05° (~5.5 km)
- **Band:** `precipitation` (mm/day)
- **Strategy:** Sum daily precipitation per month per tile, export at ~5.5 km (native). These will be used as scalar conditioning per 15 km patch (only ~3 cells across a patch).
- **Time range:** Oct 2014 – Sep 2024 (matching flood dataset)

In [ ]:
# =============================================================================
# CHIRPS Monthly Precipitation — aggregated from daily data
# Source: UCSB-CHG/CHIRPS/DAILY (0.05° ~ 5.5 km, mm/day)
# Export: monthly sum per 3x3° tile at native resolution (~5566m)
# =============================================================================

from datetime import date
from dateutil.relativedelta import relativedelta

chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')

# Flood dataset covers Oct 2014 — Sep 2024
start_date = date(2014, 10, 1)
end_date = date(2024, 9, 30)

# Generate list of (year, month) pairs
months_list = []
current = start_date
while current <= end_date:
    months_list.append((current.year, current.month))
    current += relativedelta(months=1)

print(f"Total months to process: {len(months_list)}")
print(f"First: {months_list[0]}, Last: {months_list[-1]}")

# Export parameters for CHIRPS (native resolution ~5566m = 0.05°)
chirps_export_params = {
    'scale': 5566,  # Native CHIRPS resolution in meters
    'crs': 'EPSG:4326',
    'maxPixels': 1e13,
    'fileFormat': 'GeoTIFF',
    'folder': 'Kenya_Flood_Data_3x3'
}

# Download monthly precipitation sums for each tile
for year, month in months_list:
    # Define month start/end
    m_start = f"{year}-{month:02d}-01"
    if month == 12:
        m_end = f"{year + 1}-01-01"
    else:
        m_end = f"{year}-{month + 1:02d}-01"
    
    # Filter CHIRPS to this month and sum daily values
    monthly_precip = chirps.filterDate(m_start, m_end).select('precipitation').sum()
    
    dataset_name = f"CHIRPS_precip_{year}_{month:02d}"
    
    for tile in kenya_tiles:
        download_data_for_tile(tile, monthly_precip, dataset_name, chirps_export_params)
    
    time.sleep(2)  # Brief pause between months to avoid API rate limits

print("All CHIRPS precipitation export tasks submitted.")

## ERA5-Land Monthly — Soil Moisture, Temperature, Surface Runoff
- **Source:** `ECMWF/ERA5_LAND/MONTHLY_AGGR` on GEE
- **Native resolution:** ~11 km (11132m pixel size)
- **Bands exported:**
  - `volumetric_soil_water_layer_1` (0–7 cm, volume fraction)
  - `volumetric_soil_water_layer_2` (7–28 cm, volume fraction)
  - `volumetric_soil_water_layer_3` (28–100 cm, volume fraction)
  - `volumetric_soil_water_layer_4` (100–289 cm, volume fraction)
  - `temperature_2m` (K, monthly mean)
  - `surface_runoff_sum` (m, monthly total)
  - `runoff_sum` (m, monthly total — surface + subsurface)
  - `total_precipitation_sum` (m, monthly total — ERA5's own precip for cross-validation with CHIRPS)
  - `total_evaporation_sum` (m, monthly total — for P−ET balance)
- **Strategy:** Export at native ~11 km resolution per 3°×3° tile per month. These are scalar conditioning variables (only ~1–2 cells across a 15 km patch).
- **Time range:** Oct 2014 – Sep 2024

In [ ]:
# =============================================================================
# ERA5-Land Monthly — Soil Moisture (4 layers)
# Source: ECMWF/ERA5_LAND/MONTHLY_AGGR (~11 km = 11132m)
# Export: per 3x3° tile per month at native resolution
# =============================================================================

era5_land = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')

# Soil moisture bands (volume fraction, 4 depth layers)
soil_moisture_bands = [
    'volumetric_soil_water_layer_1',  # 0-7 cm
    'volumetric_soil_water_layer_2',  # 7-28 cm
    'volumetric_soil_water_layer_3',  # 28-100 cm
    'volumetric_soil_water_layer_4',  # 100-289 cm
]

# Export parameters for ERA5-Land (native resolution ~11132m)
era5_export_params = {
    'scale': 11132,  # Native ERA5-Land resolution
    'crs': 'EPSG:4326',
    'maxPixels': 1e13,
    'fileFormat': 'GeoTIFF',
    'folder': 'Kenya_Flood_Data_3x3'
}

# Download monthly soil moisture for each tile
for year, month in months_list:
    m_start = f"{year}-{month:02d}-01"
    if month == 12:
        m_end = f"{year + 1}-01-01"
    else:
        m_end = f"{year}-{month + 1:02d}-01"
    
    # ERA5-Land monthly has one image per month — filter and get first
    era5_month = era5_land.filterDate(m_start, m_end).first()
    
    # Select soil moisture bands as multi-band image
    soil_moisture_img = era5_month.select(soil_moisture_bands)
    
    dataset_name = f"ERA5_soil_moisture_{year}_{month:02d}"
    
    for tile in kenya_tiles:
        download_data_for_tile(tile, soil_moisture_img, dataset_name, era5_export_params)
    
    time.sleep(2)

print("All ERA5-Land soil moisture export tasks submitted.")

In [ ]:
# =============================================================================
# ERA5-Land Monthly — Temperature (2m)
# Source: ECMWF/ERA5_LAND/MONTHLY_AGGR (~11 km = 11132m)
# Band: temperature_2m (K, monthly mean)
# =============================================================================

for year, month in months_list:
    m_start = f"{year}-{month:02d}-01"
    if month == 12:
        m_end = f"{year + 1}-01-01"
    else:
        m_end = f"{year}-{month + 1:02d}-01"
    
    era5_month = era5_land.filterDate(m_start, m_end).first()
    
    temp_img = era5_month.select('temperature_2m')
    
    dataset_name = f"ERA5_temperature_2m_{year}_{month:02d}"
    
    for tile in kenya_tiles:
        download_data_for_tile(tile, temp_img, dataset_name, era5_export_params)
    
    time.sleep(2)

print("All ERA5-Land temperature export tasks submitted.")

In [ ]:
# =============================================================================
# ERA5-Land Monthly — Surface Runoff, Total Runoff, Precipitation, Evapotranspiration
# Source: ECMWF/ERA5_LAND/MONTHLY_AGGR (~11 km = 11132m)
# Bands (all monthly sums in meters of water equivalent):
#   - surface_runoff_sum: surface runoff (overland flow — direct flood driver)
#   - runoff_sum: total runoff (surface + sub-surface)
#   - total_precipitation_sum: ERA5 precipitation (cross-reference with CHIRPS)
#   - total_evaporation_sum: total ET (for P-ET balance computation)
# =============================================================================

runoff_bands = [
    'surface_runoff_sum',
    'runoff_sum',
    'total_precipitation_sum',
    'total_evaporation_sum',
]

for year, month in months_list:
    m_start = f"{year}-{month:02d}-01"
    if month == 12:
        m_end = f"{year + 1}-01-01"
    else:
        m_end = f"{year}-{month + 1:02d}-01"
    
    era5_month = era5_land.filterDate(m_start, m_end).first()
    
    runoff_img = era5_month.select(runoff_bands)
    
    dataset_name = f"ERA5_runoff_precip_et_{year}_{month:02d}"
    
    for tile in kenya_tiles:
        download_data_for_tile(tile, runoff_img, dataset_name, era5_export_params)
    
    time.sleep(2)

print("All ERA5-Land runoff/precip/ET export tasks submitted.")

In [ ]:
# Check status of all export tasks
check_task_status(n=100)